# 강의 03 · 실습 2 — 랭체인 기초 · (3) 변형

## 1. 문제상황

- 온라인 서점 고객센터는 하루에 수십 건의 책 리뷰를 받습니다.
- 담당자는 리뷰마다 감정이 긍정인지 부정인지 판단하고, 핵심어를 뽑아 표에 적고, 답글 초안을 써서 게시판에 올립니다.
- 리뷰 정리 결과는 게시판 형식에 맞춘 한 덩어리 글(카드)로 붙여 넣어야 하는데, 지금은 판단 결과를 손으로 옮겨 적어 카드를 만듭니다.
- 리뷰 수가 늘면 판단과 옮겨 적기가 그만큼 반복됩니다.

## 2. 문제와 목표

- **문제**: 리뷰의 감정 판단, 핵심어 추출, 답글 초안 작성, 게시판 카드 만들기를 사람이 리뷰마다 반복합니다.
- **목표**
  - 리뷰 한 건을 넣으면 감정·핵심어·답글 초안을 정해진 모양으로 받습니다.
    - 정해진 모양(스키마 세 칸): 감정(긍정·부정·중립 중 하나), 핵심어(문자열 리스트), 답글 초안(문자열)
  - 그 결과를 게시판 카드 문자열로 바꿔 돌려주는 체인을 만듭니다.
    - 카드: 감정 표시·핵심어·답글 세 줄
- **목표 달성 여부의 판정 기준**:
  - 긍정 리뷰와 부정 리뷰를 각각 넣었을 때, 감정이 리뷰에 맞게 「긍정」·「부정」으로 판정되고,
  - 핵심어가 리스트로 2개 이상 나오며, 카드 문자열에 감정 표시·핵심어·답글이 모두 들어 있는 것을 실행 기록에서 확인합니다.

## 3. 워크플로우 다이어그램

![워크플로우 다이어그램](imgs/lec03_ex02_s3_diagram.svg)

## 4. 단계별 요구사항

1. **출력 스키마를 선언합니다.**
    - `sentiment`(감정, `Literal["긍정", "부정", "중립"]` 중 하나), `keywords`(핵심어, 문자열 리스트 `list[str]`), `reply`(답글 초안, 문자열) 세 칸을 가지는 `ReviewSummary` 클래스를 선언합니다.
2. **모델을 초기화합니다.**
    - `init_chat_model("openai/gpt-5.6-luna", model_provider="litellm")`로 모델 부품 `llm`을 만듭니다.
3. **프롬프트 템플릿을 선언합니다.**
    - 시스템 메시지(「너는 온라인 서점 고객센터 담당자다. 리뷰를 읽고 감정, 핵심어 2개 이상, 두 문장 답글 초안을 만든다.」)와 사용자 메시지(`{review}` 빈칸)를 가지는 템플릿을 만듭니다.
    - 빈칸 이름은 실행할 때 넣는 키 `review`와 같아야 합니다.
4. **카드 함수를 만들고 부품을 조립합니다.**
    - `ReviewSummary` 객체를 받아 게시판 카드 문자열을 돌려주는 함수 `to_card`를 만듭니다.
    - 카드는 감정 표시(`{"긍정": "[+]", "부정": "[-]", "중립": "[0]"}` 사전에서 찾음), 핵심어(`", ".join(keywords)`), 답글 초안을 세 줄로 담습니다.
    - `prompt | llm.with_structured_output(ReviewSummary) | to_card`로 체인을 만듭니다.
5. **실행합니다.**
    - 긍정 리뷰(「배송이 빨랐고 종이 질도 좋아요. 삽화가 많아 아이가 좋아합니다.」)와 부정 리뷰(「표지가 찢어진 채로 왔고 문의 답변도 이틀째 없네요.」)를 차례로 넣어 카드 문자열을 출력합니다.

## 5. 코드 골격 — LangChain 체인 3단

부품 선언, 조립, 실행의 세 단계입니다. 조립 단계에서 체인 끝에 파이썬 함수를 부품으로 잇습니다.

| 단계 | 하는 일 | 사용하는 코드 | 대응하는 요구사항 |
|---|---|---|---|
| ① 부품 선언 | 모델·프롬프트·출력 파서를 각각 하나씩 선언합니다 | `class ReviewSummary(BaseModel)`, `init_chat_model(...)`, `ChatPromptTemplate.from_messages` | 1, 2, 3 |
| ② 조립 | 부품을 파이프 기호로 한 줄에 잇습니다 | `prompt | llm.with_structured_output(ReviewSummary) | to_card` | 4 |
| ③ 실행 | 입력을 넣어 체인을 돌리고, 필요하면 조각으로 받습니다 | `chain.invoke({"review": ...})` | 5 |

## 6. 코드 — 스텝바이스텝

### 단계 0 — 준비

라이브러리를 불러오고 API 키를 읽습니다.

- API 키는 `.env` 파일에서 읽습니다.
- `.env` 파일은 실습 루트 폴더(`agentic-ai`)에 한 개만 둡니다. `find_dotenv()`가 노트북 위치에서 상위 폴더로 올라가며 찾습니다.
- `.env` 파일에는 다음 한 줄만 넣습니다.

```
OPENAI_API_KEY=발급받은_키
```

In [ ]:
import os

from dotenv import load_dotenv, find_dotenv
from typing import Literal

from pydantic import BaseModel

from langchain.chat_models import init_chat_model
from langchain_core.prompts import ChatPromptTemplate

load_dotenv(find_dotenv(usecwd=True))
if not os.environ.get("OPENAI_API_KEY"):
    raise SystemExit("agentic-ai 폴더의 .env 파일에 OPENAI_API_KEY 한 줄을 넣습니다.")
print("준비를 마쳤습니다.")

### 단계 ① — 부품 선언 (요구사항 1, 2, 3)

- 스키마의 칸에는 타입을 정확히 적습니다. `Literal`은 값을 정해진 선택지로 제한하고, `list[str]`은 리스트로 받게 합니다. 이 타입이 뒤 부품이 읽는 값의 모양을 정합니다.
- 템플릿의 빈칸 이름(`{review}`)은 실행할 때 넣는 딕셔너리의 키와 같아야 합니다.

In [ ]:
# 여기에 단계 ①(출력 스키마, 모델, 프롬프트 템플릿 선언)을 작성합니다.

### 단계 ② — 조립 (요구사항 4)

- 파이썬 함수는 입력 하나를 받아 값 하나를 돌려주기만 하면 부품입니다. 앞 부품의 출력(`ReviewSummary` 객체)이 함수의 인자가 되고, 함수의 반환값이 체인의 결과가 됩니다.
- 함수가 읽는 칸 이름·타입·값 집합은 스키마와 같아야 합니다. 어긋나도 파이썬은 오류를 내지 않고 이상한 값을 만들 수 있습니다.

In [ ]:
# 여기에 단계 ②(카드 함수와 체인 조립)를 작성합니다.

### 단계 ③ — 실행 (요구사항 5)

- 입력 딕셔너리의 키 `review`가 템플릿의 빈칸으로 들어갑니다.
- 체인의 결과는 마지막 부품인 카드 함수의 반환값, 즉 문자열입니다.

In [ ]:
# 여기에 단계 ③(리뷰 두 건 실행)을 작성합니다.

## 7. 실행 결과 확인

셀을 위에서 아래로 모두 실행한 뒤 다음 세 가지를 확인합니다.

1. 단계 ①의 출력에서 스키마 칸이 `['sentiment', 'keywords', 'reply']`이고 템플릿 빈칸이 `['review']`입니다.
2. 1번 리뷰의 카드 첫 줄이 `[+] 감정: 긍정`, 2번 리뷰의 카드 첫 줄이 `[-] 감정: 부정`입니다. 감정 표시가 `None`이 아닙니다.
3. 두 카드의 `핵심어:` 줄에 단어가 쉼표로 2개 이상 이어져 있고(글자 하나씩 쉼표로 끊긴 모양이 아님), `답글:` 줄이 리뷰 내용에 맞는 두 문장입니다.

세 가지가 모두 확인되면 완성입니다.